# Raspagem de dados da Web + Análise de Sentimentos

T2ESOFT06NA · Ciência de Dados · 202623

Roteiro: **T1** escolher o site → T2 raspar os comentários → T3 rotular e pré-processar →
T4 treinar e avaliar → T5 resultados e insights.

## T1 · Escolher o site

Para cada loja candidata:
1. abra no navegador uma página de produto que tenha comentários **com nota em estrelas**;
2. copie a URL e um trecho curto (5 a 8 palavras) de um comentário que aparece na tela;
3. cole os dois na célula `CANDIDATOS` abaixo e rode as células seguintes.

A célula de teste confere o `robots.txt`, baixa a página com `requests` (sem JavaScript) e procura o
trecho no HTML. Se o trecho estiver lá, ela mostra as tags e classes em volta dele, que é o que você vai
usar no `find_all` (igual ao notebook de referência, em que se inspeciona a tag e a classe no HTML).

Fica o **primeiro** candidato em que o trecho aparece e a nota também pode ser extraída (decisão D2).
Se nenhum passar, use o **plano B** mais abaixo.

In [ ]:
import re
import time
from pathlib import Path
from urllib.parse import urlsplit
from urllib.robotparser import RobotFileParser

import requests
from bs4 import BeautifulSoup

CABECALHOS = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
PASTA_PAGINAS = Path('paginas')
PASTA_PAGINAS.mkdir(exist_ok=True)

In [ ]:
# Preencha 2 ou 3 candidatos. Linhas com 'COLE_AQUI' são ignoradas.
CANDIDATOS = [
    {'loja': 'loja1', 'url': 'COLE_AQUI_A_URL_DO_PRODUTO', 'trecho': 'COLE_AQUI_UM_TRECHO_DO_COMENTARIO'},
    {'loja': 'loja2', 'url': 'COLE_AQUI_A_URL_DO_PRODUTO', 'trecho': 'COLE_AQUI_UM_TRECHO_DO_COMENTARIO'},
    {'loja': 'loja3', 'url': 'COLE_AQUI_A_URL_DO_PRODUTO', 'trecho': 'COLE_AQUI_UM_TRECHO_DO_COMENTARIO'},
]

In [ ]:
def permitido(url):
    """True/False conforme o robots.txt do site; None se não deu para ler o arquivo."""
    partes = urlsplit(url)
    url_robots = f'{partes.scheme}://{partes.netloc}/robots.txt'
    try:
        r = requests.get(url_robots, headers=CABECALHOS, timeout=20)
    except requests.RequestException as erro:
        print(f'  robots.txt: falha ao baixar ({erro})')
        return None
    if r.status_code == 404:
        return True  # sem robots.txt, nada é proibido
    if r.status_code != 200:
        print(f'  robots.txt: HTTP {r.status_code}')
        return None
    rp = RobotFileParser()
    rp.parse(r.text.splitlines())
    return rp.can_fetch('*', url)


def decodificar(conteudo):
    """Bytes -> texto: tenta UTF-8 e cai para latin1 (cuidado com acentos do notebook de referência)."""
    try:
        return conteudo.decode('utf-8')
    except UnicodeDecodeError:
        return conteudo.decode('latin1')


def baixar(url, arquivo):
    """Baixa a página e salva em disco, para o parser poder ser testado sem acessar o site de novo."""
    r = requests.get(url, headers=CABECALHOS, timeout=30)
    r.raise_for_status()
    html = decodificar(r.content)
    arquivo.write_text(html, encoding='utf-8')
    return html


def procurar_trecho(html, trecho, niveis=4):
    """Mostra onde o trecho aparece no HTML: a tag de cada ancestral e suas classes."""
    soup = BeautifulSoup(html, 'html.parser')
    achados = soup.find_all(string=lambda s: s and trecho.lower() in s.lower())
    for i, texto in enumerate(achados[:3], 1):
        print(f'  ocorrência {i}: {texto.strip()[:80]!r}')
        tag = texto.parent
        for _ in range(niveis):
            if tag is None or tag.name == '[document]':
                break
            print(f'    <{tag.name}> class={tag.get("class")}')
            tag = tag.parent
    return len(achados)

In [ ]:
paginas = {}  # loja -> HTML baixado

for c in CANDIDATOS:
    if 'COLE_AQUI' in c['url'] or 'COLE_AQUI' in c['trecho']:
        continue
    print(f"== {c['loja']}: {c['url']}")
    c['robots_ok'] = permitido(c['url'])
    print(f"  permitido pelo robots.txt: {c['robots_ok']}")
    if c['robots_ok'] is False:
        print('  pulado: caminho proibido no robots.txt')
        continue
    time.sleep(1)  # pausa entre requisições
    try:
        html = baixar(c['url'], PASTA_PAGINAS / f"{c['loja']}.html")
    except requests.RequestException as erro:
        print(f'  falha ao baixar a página: {erro}')
        continue
    paginas[c['loja']] = html
    n = procurar_trecho(html, c['trecho'])
    print(f'  trecho encontrado no HTML: {n} vez(es)')
    print('  -> candidato viável' if n else '  -> o comentário não vem no HTML (provavelmente carregado por JavaScript)')
    time.sleep(1)

### Plano B · página salva no navegador

Use se nenhum candidato acima trouxe o comentário no HTML.
1. Abra a página do produto no navegador e role até os comentários ficarem **visíveis na tela**
   (clique em "ver mais avaliações", se houver).
2. Salve a página (Ctrl+S).
3. No Colab, envie o arquivo `.html` pelo painel **Arquivos** (ícone de pasta, à esquerda).
4. Escreva o nome do arquivo em `PAGINA_SALVA` e o trecho do comentário em `TRECHO_SALVO`.

In [ ]:
PAGINA_SALVA = ''   # ex.: 'produto.html'; deixe vazio se não usar o plano B
TRECHO_SALVO = ''

if PAGINA_SALVA:
    html = decodificar(Path(PAGINA_SALVA).read_bytes())
    paginas['plano_b'] = html
    n = procurar_trecho(html, TRECHO_SALVO)
    print(f'trecho encontrado no arquivo salvo: {n} vez(es)')

### Seletores do site escolhido

Com a saída da busca acima (e o "Inspecionar" do navegador), preencha a tag e a classe de:
- **bloco**: o elemento que envolve um comentário inteiro (texto + nota);
- **texto**: o elemento, dentro do bloco, com o texto do comentário;
- **nota**: o elemento, dentro do bloco, que traz a nota. A nota pode estar no texto (ex.: "4 de 5")
  ou num atributo como `aria-label`, `title` ou `data-rating`; a função procura nos dois.

Use a classe que se repete em todos os comentários, não uma classe única de um só.

In [ ]:
SITE_ESCOLHIDO = ''   # nome da loja em CANDIDATOS, ou 'plano_b'
URL_ESCOLHIDA = ''    # URL da página do produto (também no plano B, para conferir o robots.txt)
SELETORES = {
    'bloco': ('div', 'CLASSE_DO_BLOCO'),
    'texto': ('p', 'CLASSE_DO_TEXTO'),
    'nota':  ('span', 'CLASSE_DA_NOTA'),
}

PADRAO_NOTA = re.compile(r'([1-5])(?:[.,]\d+)?\s*(?:de|/|out of)\s*5', re.IGNORECASE)


def ler_nota(tag):
    """Procura 'N de 5' no texto e nos atributos; senão, aceita um número de 1 a 5 sozinho."""
    if tag is None:
        return None
    fontes = [tag.get_text(' ', strip=True)] + [str(tag.get(a, '')) for a in ('aria-label', 'title', 'data-rating')]
    for fonte in fontes:
        m = PADRAO_NOTA.search(fonte)
        if m:
            return int(m.group(1))
    for fonte in fontes:
        m = re.fullmatch(r'\s*([1-5])(?:[.,]0+)?\s*', fonte)
        if m:
            return int(m.group(1))
    return None


def extrair_comentarios(html):
    soup = BeautifulSoup(html, 'html.parser')
    linhas = []
    for bloco in soup.find_all(SELETORES['bloco'][0], class_=SELETORES['bloco'][1]):
        texto = bloco.find(SELETORES['texto'][0], class_=SELETORES['texto'][1])
        nota = ler_nota(bloco.find(SELETORES['nota'][0], class_=SELETORES['nota'][1]))
        if texto is not None and nota is not None:
            linhas.append({'nota': nota, 'comentario': texto.get_text(' ', strip=True)})
    return linhas

In [ ]:
# Aceite do T1
assert SITE_ESCOLHIDO in paginas, 'Preencha SITE_ESCOLHIDO com uma loja que foi baixada (ou plano_b).'
assert permitido(URL_ESCOLHIDA) is True, 'O robots.txt não permite (ou não confirmou) este caminho.'

comentarios = extrair_comentarios(paginas[SITE_ESCOLHIDO])
assert len(comentarios) >= 1, 'Nenhum comentário com texto e nota: revise SELETORES.'
print(f'{len(comentarios)} comentário(s) extraído(s). Primeiros:')
for c in comentarios[:3]:
    print(f"  [{c['nota']}] {c['comentario'][:120]}")